# LeetCode 1609 — Even Odd Tree bằng Python

Link đề: https://leetcode.com/problems/even-odd-tree/ · Độ khó: **Medium** · Chủ đề: Tree, Breadth-First Search, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def isEvenOddTree(self, root: Optional[TreeNode]) -> bool:
        ...
```

## 1. Mục tiêu học

1. Dựng cây từ mảng level-order và **liệt kê các tầng** của cây.
2. Thành thạo **BFS theo tầng**, so sánh mỗi nút với **nút liền trước cùng tầng** và **reset** biến "nút trước" ở đầu mỗi tầng.
3. Viết lời giải **DFS** dùng mảng `last[level]` lưu giá trị cuối cùng đã thấy ở mỗi tầng (duyệt trái trước).
4. Gộp hai điều kiện (chẵn/lẻ và tăng/giảm nghiêm ngặt) thành một quy tắc gọn, tránh nhầm chiều.
5. Kiểm thử bằng cây ngẫu nhiên được **sinh hợp lệ có chủ đích** rồi gây lỗi ngẫu nhiên, đối chiếu nhiều cách.

## 2. Đề bài & ví dụ

Một cây nhị phân gọi là **Even-Odd** nếu:
- Gốc ở tầng chỉ số 0, các con của nó ở tầng 1, cháu ở tầng 2, …
- Với mọi tầng chỉ số **chẵn**: mọi nút có giá trị **lẻ**, và **tăng nghiêm ngặt** từ trái sang phải.
- Với mọi tầng chỉ số **lẻ**: mọi nút có giá trị **chẵn**, và **giảm nghiêm ngặt** từ trái sang phải.

Cho gốc `root`, trả về `True` nếu cây là Even-Odd, ngược lại `False`.

**Ví dụ 1:** `root = [1,10,4,3,null,7,9,12,8,6,null,null,2]` → `True`. Các tầng: `[1]`, `[10,4]`, `[3,7,9]`, `[12,8,6,2]`.

**Ví dụ 2:** `root = [5,4,2,3,3,7]` → `False`. Tầng 2 là `[3,3,7]` — không tăng nghiêm ngặt.

**Ví dụ 3:** `root = [5,9,1,3,5,7]` → `False`. Tầng 1 phải gồm số chẵn.

**Ràng buộc:**
- Số nút trong khoảng `[1, 10⁵]`.
- `1 ≤ Node.val ≤ 10⁶`.

In [1]:
import random
import time
from collections import deque
from typing import Optional, List

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(arr):
    """Dựng cây từ mảng level-order kiểu LeetCode (None = không có nút)."""
    if not arr or arr[0] is None:
        return None
    root = TreeNode(arr[0]); q = deque([root]); i = 1
    while q and i < len(arr):
        node = q.popleft()
        if i < len(arr) and arr[i] is not None:
            node.left = TreeNode(arr[i]); q.append(node.left)
        i += 1
        if i < len(arr) and arr[i] is not None:
            node.right = TreeNode(arr[i]); q.append(node.right)
        i += 1
    return root

examples = [
    ([1, 10, 4, 3, None, 7, 9, 12, 8, 6, None, None, 2], True),
    ([5, 4, 2, 3, 3, 7], False),
    ([5, 9, 1, 3, 5, 7], False),
]
for arr, want in examples:
    print(arr, "-> mong đợi", want)

[1, 10, 4, 3, None, 7, 9, 12, 8, 6, None, None, 2] -> mong đợi True
[5, 4, 2, 3, 3, 7] -> mong đợi False
[5, 9, 1, 3, 5, 7] -> mong đợi False


## 3. Kiến thức nền

### 3.1 Liệt kê các tầng bằng BFS

Đầu mỗi tầng, hàng đợi chứa đúng các nút của tầng; lấy ra `len(q)` nút, đẩy con trái rồi con phải.

In [2]:
def levels(root):
    res, q = [], deque([root])
    while q:
        cur = []
        for _ in range(len(q)):
            x = q.popleft(); cur.append(x.val)
            if x.left: q.append(x.left)
            if x.right: q.append(x.right)
        res.append(cur)
    return res

for arr, _ in examples:
    print(levels(build_tree(arr)))

[[1], [10, 4], [3, 7, 9], [12, 8, 6, 2]]
[[5], [4, 2], [3, 3, 7]]
[[5], [9, 1], [3, 5, 7]]


### 3.2 Chẵn / lẻ với `%`

`v % 2 == 1` ⇔ `v` lẻ. Tầng `d` chẵn ⇔ `d % 2 == 0`. Đề yêu cầu: **tầng chẵn chứa số lẻ, tầng lẻ chứa số chẵn** ⇒ luôn có `v % 2 != d % 2`. Ngược lại, nếu `v % 2 == d % 2` thì vi phạm.

In [3]:
for d, v in [(0, 1), (0, 2), (1, 10), (1, 9)]:
    print(f"tầng {d}, giá trị {v}: {'hợp lệ' if v % 2 != d % 2 else 'SAI chẵn/lẻ'}")

tầng 0, giá trị 1: hợp lệ
tầng 0, giá trị 2: SAI chẵn/lẻ
tầng 1, giá trị 10: hợp lệ
tầng 1, giá trị 9: SAI chẵn/lẻ


### 3.3 Tăng / giảm **nghiêm ngặt**

"Strictly increasing": `a < b` (không cho bằng). "Strictly decreasing": `a > b`. Hàm kiểm tra một dãy:

In [4]:
def strictly_inc(a):
    return all(a[i] < a[i + 1] for i in range(len(a) - 1))
def strictly_dec(a):
    return all(a[i] > a[i + 1] for i in range(len(a) - 1))
print(strictly_inc([3, 7, 9]), strictly_inc([3, 3, 7]), strictly_dec([12, 8, 6, 2]), strictly_dec([4, 4]))

True False True False


### 3.4 Mảng giá trị cuối theo tầng cho DFS

Khi DFS **trái trước**, các nút của một tầng được gặp theo thứ tự trái → phải. Lưu `last[d]` = giá trị nút gần nhất đã gặp ở tầng `d`; khi gặp nút mới ở tầng `d`, so sánh với `last[d]`. Nếu `d == len(last)` thì đây là nút đầu tiên của tầng: `last.append(v)`.

In [5]:
def dfs_order(root):
    out, st = [], [(root, 0)]
    while st:
        x, d = st.pop()
        out.append((d, x.val))
        if x.right: st.append((x.right, d + 1))
        if x.left: st.append((x.left, d + 1))
    return out
print(dfs_order(build_tree(examples[0][0])))

[(0, 1), (1, 10), (2, 3), (3, 12), (3, 8), (1, 4), (2, 7), (3, 6), (2, 9), (3, 2)]


## 4. Ý tưởng nền tảng / trực giác

Kiểm tra tay ví dụ 1 theo từng tầng:

| tầng | giá trị | chẵn/lẻ yêu cầu | thứ tự yêu cầu | đạt? |
|---|---|---|---|---|
| 0 | [1] | lẻ | tăng | ✓ |
| 1 | [10, 4] | chẵn | giảm | ✓ |
| 2 | [3, 7, 9] | lẻ | tăng | ✓ |
| 3 | [12, 8, 6, 2] | chẵn | giảm | ✓ |

Ví dụ 2: tầng 2 `[3, 3, 7]` — `3 < 3` sai ⇒ `False`. Ví dụ 3: tầng 1 `[9, 1]` gồm số lẻ ⇒ `False`.

**Nhận xét:** điều kiện chỉ liên quan đến **các nút cùng tầng, theo thứ tự trái → phải**, và mỗi nút chỉ cần so với **nút liền trước** trong tầng (tính chất "tăng nghiêm ngặt" của cả dãy ⇔ mọi cặp liền kề tăng).

In [6]:
for arr, want in examples:
    lv = levels(build_tree(arr))
    for d, vals in enumerate(lv):
        par_ok = all(v % 2 != d % 2 for v in vals)
        ord_ok = strictly_inc(vals) if d % 2 == 0 else strictly_dec(vals)
        print(f"tầng {d}: {vals} chẵn/lẻ {'✓' if par_ok else '✗'}, thứ tự {'✓' if ord_ok else '✗'}")
    print("---")

tầng 0: [1] chẵn/lẻ ✓, thứ tự ✓
tầng 1: [10, 4] chẵn/lẻ ✓, thứ tự ✓
tầng 2: [3, 7, 9] chẵn/lẻ ✓, thứ tự ✓
tầng 3: [12, 8, 6, 2] chẵn/lẻ ✓, thứ tự ✓
---
tầng 0: [5] chẵn/lẻ ✓, thứ tự ✓
tầng 1: [4, 2] chẵn/lẻ ✓, thứ tự ✓
tầng 2: [3, 3, 7] chẵn/lẻ ✓, thứ tự ✗
---
tầng 0: [5] chẵn/lẻ ✓, thứ tự ✓
tầng 1: [9, 1] chẵn/lẻ ✗, thứ tự ✓
tầng 2: [3, 5, 7] chẵn/lẻ ✓, thứ tự ✓
---


## 5. Cách 1 — Brute Force / baseline: gom mọi tầng rồi kiểm tra

**Ý tưởng:** dùng `levels(root)` lấy danh sách giá trị từng tầng; với mỗi tầng kiểm tra chẵn/lẻ và tăng/giảm. Đơn giản, đúng, nhưng luôn duyệt và lưu toàn bộ cây trước khi kiểm tra (không dừng sớm).

In [7]:
def is_even_odd_levels(root):
    for d, vals in enumerate(levels(root)):
        if d % 2 == 0:
            if any(v % 2 == 0 for v in vals) or not strictly_inc(vals):
                return False
        else:
            if any(v % 2 == 1 for v in vals) or not strictly_dec(vals):
                return False
    return True

for arr, want in examples:
    print(is_even_odd_levels(build_tree(arr)), "| mong đợi", want)

True | mong đợi True
False | mong đợi False
False | mong đợi False


**Độ phức tạp:** Time O(n), Space O(n) (lưu mọi giá trị theo tầng).

**Nút thắt:** về độ phức tạp đã tối ưu, nhưng tốn bộ nhớ lưu cả cây và không dừng khi lỗi xuất hiện ngay ở tầng đầu. Ta có thể kiểm tra **trong lúc** BFS, chỉ giữ giá trị nút trước.

In [8]:
# Minh họa: lỗi ở tầng 1 nhưng Cách 1 vẫn duyệt hết cây lớn
bad = TreeNode(1, TreeNode(3), TreeNode(2))         # tầng 1 có số lẻ 3
cur = bad.right
for v in range(3, 50003):                          # nối thêm 50k nút vào nhánh phải
    cur.right = TreeNode(v if v % 2 else v + 1); cur = cur.right
t0 = time.perf_counter(); is_even_odd_levels(bad); print(f"Cách 1 trên cây lỗi sớm: {time.perf_counter() - t0:.3f}s")

Cách 1 trên cây lỗi sớm: 0.013s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** "tăng nghiêm ngặt cả tầng" ⇔ "mỗi nút lớn hơn nút liền trước trong tầng". Nên chỉ cần **một biến `prev`** cho mỗi tầng.

**Quy tắc (tầng `d`, nút giá trị `v`, nút trước `prev`):**
- Nếu `d` chẵn: cần `v` **lẻ** và `v > prev` (khởi tạo `prev = −∞`, hoặc `0` vì `v ≥ 1`).
- Nếu `d` lẻ: cần `v` **chẵn** và `v < prev` (khởi tạo `prev = +∞`, hoặc `10⁶ + 1`).
- Vi phạm bất kỳ ⇒ trả `False` ngay.
- **Reset `prev` ở đầu mỗi tầng.**

**Gộp gọn:** với `d` chẵn, `v` lẻ và tăng; với `d` lẻ, `v` chẵn và giảm. Một cách viết: vi phạm khi `v % 2 == d % 2`, hoặc (`d` chẵn và `v <= prev`), hoặc (`d` lẻ và `v >= prev`).

**Invariant BFS:** khi xét nút thứ `k` của tầng `d`, các nút `0..k−1` của tầng đã thỏa điều kiện và `prev` là giá trị nút `k−1` (hoặc giá trị khởi tạo nếu `k = 0`).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — BFS theo tầng, kiểm tra ngay, dừng sớm

Từng đoạn:
1. `q = deque([root])`, `even = True` (tầng 0 chẵn).
2. Đầu mỗi tầng: `prev = 0 if even else inf`.
3. Lấy từng nút: kiểm tra chẵn/lẻ và thứ tự với `prev`; cập nhật `prev`; đẩy con.
4. Hết tầng: `even = not even`.

In [9]:
def is_even_odd_bfs(root):
    q = deque([root])
    even = True
    while q:
        prev = 0 if even else float("inf")
        for _ in range(len(q)):
            x = q.popleft()
            v = x.val
            if even and (v % 2 == 0 or v <= prev):
                return False
            if not even and (v % 2 == 1 or v >= prev):
                return False
            prev = v
            if x.left: q.append(x.left)
            if x.right: q.append(x.right)
        even = not even
    return True

for arr, want in examples:
    print(is_even_odd_bfs(build_tree(arr)), "| mong đợi", want)

True | mong đợi True
False | mong đợi False
False | mong đợi False


Code nộp LeetCode của Cách 2: xem mục 9.

**Độ phức tạp:** Time O(n) worst (dừng sớm khi gặp vi phạm), Space O(w) — `w` là độ rộng lớn nhất.

### 7.2 Cách 3 — DFS (trái trước) với mảng `last[level]`

Duyệt tiền thứ tự trái → phải bằng stack `(nút, tầng)`. Với mỗi nút:
1. Kiểm tra chẵn/lẻ: `v % 2 == d % 2` ⇒ `False`.
2. Nếu `d == len(last)` (nút đầu tiên của tầng): `last.append(v)`.
3. Ngược lại so với `last[d]`: tầng chẵn cần `v > last[d]`, tầng lẻ cần `v < last[d]`; rồi `last[d] = v`.

Dùng stack tường minh để không vượt giới hạn đệ quy với cây sâu `10⁵`.

In [10]:
def is_even_odd_dfs(root):
    last = []
    st = [(root, 0)]
    while st:
        x, d = st.pop()
        v = x.val
        if v % 2 == d % 2:
            return False
        if d == len(last):
            last.append(v)
        else:
            if d % 2 == 0 and v <= last[d]:
                return False
            if d % 2 == 1 and v >= last[d]:
                return False
            last[d] = v
        if x.right: st.append((x.right, d + 1))   # đẩy phải trước -> trái ra trước
        if x.left: st.append((x.left, d + 1))
    return True

for arr, want in examples:
    print(is_even_odd_dfs(build_tree(arr)), "| mong đợi", want)

True | mong đợi True
False | mong đợi False
False | mong đợi False


Vì sao `d == len(last)` đúng là "nút đầu tiên của tầng `d`"? Trong DFS, để tới một nút tầng `d` ta phải đi qua tổ tiên ở mọi tầng `0..d−1`, nên `last` đã có đủ `d` phần tử; nếu chưa có phần tử thứ `d` thì chưa nút nào của tầng `d` được thăm.

Code nộp LeetCode của Cách 3:

```python
class Solution:
    def isEvenOddTree(self, root: Optional[TreeNode]) -> bool:
        last = []
        st = [(root, 0)]
        while st:
            x, d = st.pop()
            v = x.val
            if v % 2 == d % 2:
                return False
            if d == len(last):
                last.append(v)
            else:
                if d % 2 == 0 and v <= last[d]:
                    return False
                if d % 2 == 1 and v >= last[d]:
                    return False
                last[d] = v
            if x.right:
                st.append((x.right, d + 1))
            if x.left:
                st.append((x.left, d + 1))
        return True
```

**Độ phức tạp:** Time O(n), Space O(h) cho stack + `last` (h = chiều cao).

### 7.3 Cách 4 — BFS với `(nút, tầng)` và quy tắc gộp bằng dấu

Mẹo gộp hai trường hợp: đặt `sign = 1` ở tầng chẵn, `sign = −1` ở tầng lẻ. Khi đó "tăng ở tầng chẵn, giảm ở tầng lẻ" ⇔ `sign * v > sign * prev` luôn đúng. Hàng đợi mang theo tầng nên không cần vòng `for` theo kích thước tầng; phát hiện "sang tầng mới" khi tầng của nút khác tầng đang xét.

In [11]:
def is_even_odd_sign(root):
    q = deque([(root, 0)])
    cur_level, prev = -1, None
    while q:
        x, d = q.popleft()
        v = x.val
        if v % 2 == d % 2:
            return False
        sign = 1 if d % 2 == 0 else -1
        if d != cur_level:                 # nút đầu tiên của tầng mới
            cur_level, prev = d, v
        else:
            if sign * v <= sign * prev:
                return False
            prev = v
        if x.left: q.append((x.left, d + 1))
        if x.right: q.append((x.right, d + 1))
    return True

for arr, want in examples:
    print(is_even_odd_sign(build_tree(arr)), "| mong đợi", want)

True | mong đợi True
False | mong đợi False
False | mong đợi False


Code nộp LeetCode của Cách 4:

```python
class Solution:
    def isEvenOddTree(self, root: Optional[TreeNode]) -> bool:
        q = deque([(root, 0)])
        cur_level, prev = -1, None
        while q:
            x, d = q.popleft()
            v = x.val
            if v % 2 == d % 2:
                return False
            sign = 1 if d % 2 == 0 else -1
            if d != cur_level:
                cur_level, prev = d, v
            elif sign * v <= sign * prev:
                return False
            else:
                prev = v
            if x.left:
                q.append((x.left, d + 1))
            if x.right:
                q.append((x.right, d + 1))
        return True
```

**Độ phức tạp:** Time O(n), Space O(w).

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 2 `[5,4,2,3,3,7]`:

| tầng | even? | prev khởi tạo | nút | kiểm tra | prev sau |
|---|---|---|---|---|---|
| 0 | có | 0 | 5 | lẻ ✓, 5 > 0 ✓ | 5 |
| 1 | không | ∞ | 4 | chẵn ✓, 4 < ∞ ✓ | 4 |
| 1 | không | — | 2 | chẵn ✓, 2 < 4 ✓ | 2 |
| 2 | có | 0 | 3 | lẻ ✓, 3 > 0 ✓ | 3 |
| 2 | có | — | 3 | lẻ ✓, **3 > 3 ✗** | ⇒ `False` |

In [12]:
def trace_bfs(root):
    q = deque([root]); even = True; level = 0
    while q:
        prev = 0 if even else float("inf")
        print(f"-- tầng {level} ({'chẵn: cần lẻ & tăng' if even else 'lẻ: cần chẵn & giảm'}), prev = {prev}")
        for _ in range(len(q)):
            x = q.popleft(); v = x.val
            par_ok = (v % 2 == 1) if even else (v % 2 == 0)
            ord_ok = (v > prev) if even else (v < prev)
            print(f"   nút {v}: chẵn/lẻ {'✓' if par_ok else '✗'}, so với prev={prev}: {'✓' if ord_ok else '✗'}")
            if not (par_ok and ord_ok):
                print("=> False"); return False
            prev = v
            if x.left: q.append(x.left)
            if x.right: q.append(x.right)
        even = not even; level += 1
    print("=> True"); return True

trace_bfs(build_tree(examples[1][0]))

-- tầng 0 (chẵn: cần lẻ & tăng), prev = 0
   nút 5: chẵn/lẻ ✓, so với prev=0: ✓
-- tầng 1 (lẻ: cần chẵn & giảm), prev = inf
   nút 4: chẵn/lẻ ✓, so với prev=inf: ✓
   nút 2: chẵn/lẻ ✓, so với prev=4: ✓
-- tầng 2 (chẵn: cần lẻ & tăng), prev = 0
   nút 3: chẵn/lẻ ✓, so với prev=0: ✓
   nút 3: chẵn/lẻ ✓, so với prev=3: ✗
=> False


False

In [13]:
trace_bfs(build_tree(examples[0][0]))

-- tầng 0 (chẵn: cần lẻ & tăng), prev = 0
   nút 1: chẵn/lẻ ✓, so với prev=0: ✓
-- tầng 1 (lẻ: cần chẵn & giảm), prev = inf
   nút 10: chẵn/lẻ ✓, so với prev=inf: ✓
   nút 4: chẵn/lẻ ✓, so với prev=10: ✓
-- tầng 2 (chẵn: cần lẻ & tăng), prev = 0
   nút 3: chẵn/lẻ ✓, so với prev=0: ✓
   nút 7: chẵn/lẻ ✓, so với prev=3: ✓
   nút 9: chẵn/lẻ ✓, so với prev=7: ✓
-- tầng 3 (lẻ: cần chẵn & giảm), prev = inf
   nút 12: chẵn/lẻ ✓, so với prev=inf: ✓
   nút 8: chẵn/lẻ ✓, so với prev=12: ✓
   nút 6: chẵn/lẻ ✓, so với prev=8: ✓
   nút 2: chẵn/lẻ ✓, so với prev=6: ✓
=> True


True

In [14]:
def trace_dfs(root):
    last, st = [], [(root, 0)]
    while st:
        x, d = st.pop(); v = x.val
        if v % 2 == d % 2:
            print(f"thăm {v} ở tầng {d}: sai chẵn/lẻ => False"); return False
        if d == len(last):
            last.append(v); print(f"thăm {v} ở tầng {d}: nút đầu của tầng, last = {last}")
        else:
            ok = v > last[d] if d % 2 == 0 else v < last[d]
            print(f"thăm {v} ở tầng {d}: so với last[{d}]={last[d]} -> {'✓' if ok else '✗'}")
            if not ok: print("=> False"); return False
            last[d] = v
        if x.right: st.append((x.right, d + 1))
        if x.left: st.append((x.left, d + 1))
    print("=> True, last =", last); return True

trace_dfs(build_tree(examples[0][0]))

thăm 1 ở tầng 0: nút đầu của tầng, last = [1]
thăm 10 ở tầng 1: nút đầu của tầng, last = [1, 10]
thăm 3 ở tầng 2: nút đầu của tầng, last = [1, 10, 3]
thăm 12 ở tầng 3: nút đầu của tầng, last = [1, 10, 3, 12]
thăm 8 ở tầng 3: so với last[3]=12 -> ✓
thăm 4 ở tầng 1: so với last[1]=10 -> ✓
thăm 7 ở tầng 2: so với last[2]=3 -> ✓
thăm 6 ở tầng 3: so với last[3]=8 -> ✓
thăm 9 ở tầng 2: so với last[2]=7 -> ✓
thăm 2 ở tầng 3: so với last[3]=6 -> ✓
=> True, last = [1, 4, 9, 2]


True

## 9. Code nộp LeetCode

```python
from collections import deque
from typing import Optional

class Solution:
    def isEvenOddTree(self, root: Optional[TreeNode]) -> bool:
        q = deque([root])
        even = True                              # tầng 0 là tầng chẵn
        while q:
            prev = 0 if even else float('inf')   # reset ở đầu mỗi tầng
            for _ in range(len(q)):
                x = q.popleft()
                v = x.val
                if even and (v % 2 == 0 or v <= prev):
                    return False
                if not even and (v % 2 == 1 or v >= prev):
                    return False
                prev = v
                if x.left:
                    q.append(x.left)
                if x.right:
                    q.append(x.right)
            even = not even
        return True
```

In [15]:
class Solution:
    def isEvenOddTree(self, root: Optional[TreeNode]) -> bool:
        q = deque([root])
        even = True
        while q:
            prev = 0 if even else float('inf')
            for _ in range(len(q)):
                x = q.popleft()
                v = x.val
                if even and (v % 2 == 0 or v <= prev):
                    return False
                if not even and (v % 2 == 1 or v >= prev):
                    return False
                prev = v
                if x.left:
                    q.append(x.left)
                if x.right:
                    q.append(x.right)
            even = not even
        return True

for arr, want in examples:
    got = Solution().isEvenOddTree(build_tree(arr))
    print(arr, "->", got, "OK" if got == want else "SAI")

[1, 10, 4, 3, None, 7, 9, 12, 8, 6, None, None, 2] -> True OK
[5, 4, 2, 3, 3, 7] -> False OK
[5, 9, 1, 3, 5, 7] -> False OK


## 10. Vì sao thuật toán đúng

1. **BFS liệt kê đúng từng tầng theo thứ tự trái → phải** (quy nạp: tầng 0 chỉ có gốc; nếu tầng `d` đúng thứ tự thì việc đẩy con trái rồi con phải của từng nút theo thứ tự cho tầng `d+1` đúng thứ tự).
2. **Điều kiện cục bộ ⇔ điều kiện toàn cục:** một dãy tăng nghiêm ngặt ⇔ mọi cặp liền kề tăng nghiêm ngặt (tính bắc cầu của `<`). Tương tự với giảm. Điều kiện chẵn/lẻ là theo từng nút. Vậy kiểm tra từng nút với nút liền trước trong tầng là đủ.
3. **Giá trị khởi tạo `prev`:** `0` nhỏ hơn mọi giá trị hợp lệ (`v ≥ 1`), `∞` lớn hơn mọi giá trị ⇒ nút đầu tầng chỉ bị kiểm tra chẵn/lẻ.
4. **Dừng sớm an toàn:** một vi phạm đủ để kết luận `False`; nếu duyệt hết không vi phạm thì mọi tầng thỏa ⇒ `True`.
5. **DFS (Cách 3):** duyệt trái trước gặp các nút của mỗi tầng theo thứ tự trái → phải, nên `last[d]` luôn là nút liền trước trong tầng.

## 11. Phân tích Time / Space

Gọi `n` = số nút, `h` = chiều cao, `w` = độ rộng lớn nhất.

| Cách | Time | Space |
|---|---|---|
| 1. Gom tầng | O(n) luôn | O(n) |
| 2. BFS dừng sớm | O(n) worst | O(w) |
| 3. DFS + last[] | O(n) worst | O(h) |
| 4. BFS (nút, tầng) + dấu | O(n) worst | O(w) |

Đo với cây hợp lệ `10⁵` nút (trường hợp phải duyệt hết):

In [16]:
def random_shape(n, rnd):
    nodes = [TreeNode(0) for _ in range(n)]
    free = [(nodes[0], 0), (nodes[0], 1)]
    for x in nodes[1:]:
        k = rnd.randrange(len(free)); p, side = free[k]
        free[k] = free[-1]; free.pop()
        if side == 0: p.left = x
        else: p.right = x
        free += [(x, 0), (x, 1)]
    return nodes[0]

def assign_valid_values(root, rnd, maxv=10**6):
    """Gán giá trị để cây là Even-Odd: tầng chẵn số lẻ tăng dần, tầng lẻ số chẵn giảm dần."""
    q, d = deque([root]), 0
    while q:
        lv = [q.popleft() for _ in range(len(q))]
        pool = range(1, maxv + 1, 2) if d % 2 == 0 else range(2, maxv + 1, 2)
        vals = sorted(rnd.sample(pool, len(lv)), reverse=(d % 2 == 1))
        for x, v in zip(lv, vals):
            x.val = v
            if x.left: q.append(x.left)
            if x.right: q.append(x.right)
        d += 1
    return root

rnd = random.Random(0)
big = assign_valid_values(random_shape(10**5, rnd), rnd)
for f in [is_even_odd_levels, is_even_odd_bfs, is_even_odd_dfs, is_even_odd_sign]:
    t0 = time.perf_counter(); r = f(big)
    print(f"{f.__name__:20s} -> {r}, {time.perf_counter() - t0:.3f}s")

is_even_odd_levels   -> True, 0.026s
is_even_odd_bfs      -> True, 0.022s
is_even_odd_dfs      -> True, 0.031s
is_even_odd_sign     -> True, 0.037s


## 12. Bộ test

Ví dụ đề, edge case (1 nút lẻ / chẵn, tầng có giá trị bằng nhau, đúng thứ tự nhưng sai chẵn/lẻ, đúng chẵn/lẻ nhưng sai thứ tự, cây lệch sâu), và test ngẫu nhiên: sinh cây **hợp lệ** rồi (một nửa số lần) đổi giá trị một nút để phá vỡ — đối chiếu mọi cách với Cách 1.

In [17]:
ALL = [is_even_odd_levels, is_even_odd_bfs, is_even_odd_dfs, is_even_odd_sign,
       lambda r: Solution().isEvenOddTree(r)]
fixed = examples + [
    ([1], True), ([2], False),
    ([1, 4, 2], True), ([1, 2, 4], False),          # tầng 1 phải giảm
    ([1, 4, 4], False),                              # bằng nhau
    ([1, 4, 2, 3, 5], True), ([1, 4, 2, 5, 3], False),
    ([1, 3, 1], False),                              # tầng 1 phải chẵn
    ([11, 8, 6, 1, 3, 5, 7], True),
    ([1, 10, None, 3, None, 4], True),               # lệch trái
    ([1, None, 2, None, 3], True),                   # lệch phải
]
for arr, want in fixed:
    for f in ALL:
        assert f(build_tree(arr)) == want, (f, arr)
print("Ca cố định:", len(fixed), "OK")

rnd = random.Random(2024)
stat = {True: 0, False: 0}
for _ in range(1500):
    n = rnd.randint(1, 40)
    root = assign_valid_values(random_shape(n, rnd), rnd, maxv=200)
    if rnd.random() < 0.5:                           # phá vỡ: đổi giá trị một nút ngẫu nhiên
        nodes, st = [], [root]
        while st:
            x = st.pop(); nodes.append(x)
            st += [c for c in (x.left, x.right) if c]
        rnd.choice(nodes).val = rnd.randint(1, 200)
    want = is_even_odd_levels(root)
    stat[want] += 1
    for f in ALL[1:]:
        assert f(root) == want, f
print("Ngẫu nhiên: OK, phân bố kết quả", stat)

Ca cố định: 14 OK
Ngẫu nhiên: OK, phân bố kết quả {True: 954, False: 546}


In [18]:
# Cây lệch sâu 1e5 hợp lệ (mỗi tầng 1 nút, giá trị xen kẽ lẻ/chẵn): các cách không đệ quy chạy tốt
root = TreeNode(1); cur = root
for d in range(1, 10**5):
    cur.left = TreeNode(2 * d + (0 if d % 2 else 1)); cur = cur.left
for f in ALL:
    assert f(root) is True
cur.val += 1                                          # phá chẵn/lẻ ở tầng cuối
for f in ALL:
    assert f(root) is False
print("Cây lệch sâu 1e5: OK")

Cây lệch sâu 1e5: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — So sánh không nghiêm ngặt (`<` thay vì `<=` ở điều kiện vi phạm)

In [19]:
def wrong_non_strict(root):
    q = deque([root]); even = True
    while q:
        prev = 0 if even else float("inf")
        for _ in range(len(q)):
            x = q.popleft(); v = x.val
            if even and (v % 2 == 0 or v < prev):       # SAI: cho phép v == prev
                return False
            if not even and (v % 2 == 1 or v > prev):   # SAI
                return False
            prev = v
            if x.left: q.append(x.left)
            if x.right: q.append(x.right)
        even = not even
    return True

r = build_tree(examples[1][0])
print("sai :", wrong_non_strict(r), "| đúng:", Solution().isEvenOddTree(r))

sai : True | đúng: False


**Vì sao sai:** tầng 2 `[3, 3, 7]` có hai giá trị bằng nhau, "strictly increasing" không cho phép. **Sửa:** vi phạm khi `v <= prev` (tầng chẵn) / `v >= prev` (tầng lẻ).

### Lỗi 2 — Nhầm chiều điều kiện chẵn/lẻ

In [20]:
def wrong_parity(root):
    q = deque([root]); even = True
    while q:
        prev = 0 if even else float("inf")
        for _ in range(len(q)):
            x = q.popleft(); v = x.val
            if even and (v % 2 == 1 or v <= prev):      # SAI: tưởng tầng chẵn cần số chẵn
                return False
            if not even and (v % 2 == 0 or v >= prev):
                return False
            prev = v
            if x.left: q.append(x.left)
            if x.right: q.append(x.right)
        even = not even
    return True

r = build_tree(examples[0][0])
print("sai :", wrong_parity(r), "| đúng:", Solution().isEvenOddTree(r))

sai : False | đúng: True


**Vì sao sai:** đề ngược với trực giác: tầng chỉ số **chẵn** chứa số **lẻ**, tầng **lẻ** chứa số **chẵn**. Mẹo nhớ: luôn `v % 2 != d % 2`. **Sửa:** như Cách 2.

### Lỗi 3 — Không reset `prev` ở đầu mỗi tầng

In [21]:
def wrong_no_reset(root):
    q = deque([root]); even = True
    prev = 0                                            # SAI: chỉ khởi tạo một lần
    while q:
        for _ in range(len(q)):
            x = q.popleft(); v = x.val
            if even and (v % 2 == 0 or v <= prev):
                return False
            if not even and (v % 2 == 1 or v >= prev):
                return False
            prev = v
            if x.left: q.append(x.left)
            if x.right: q.append(x.right)
        even = not even
    return True

r = build_tree(examples[0][0])
print("sai :", wrong_no_reset(r), "| đúng:", Solution().isEvenOddTree(r))

sai : False | đúng: True


**Vì sao sai:** nút `10` (đầu tầng 1) bị so với `prev = 1` từ tầng 0 và yêu cầu `10 < 1` ⇒ báo sai. Mỗi tầng là một dãy độc lập. **Sửa:** đặt lại `prev` (0 hoặc ∞) ở đầu mỗi tầng.

### Lỗi 4 — DFS thăm con phải trước

In [22]:
def wrong_dfs_right_first(root):
    last, st = [], [(root, 0)]
    while st:
        x, d = st.pop(); v = x.val
        if v % 2 == d % 2: return False
        if d == len(last): last.append(v)
        else:
            if d % 2 == 0 and v <= last[d]: return False
            if d % 2 == 1 and v >= last[d]: return False
            last[d] = v
        if x.left: st.append((x.left, d + 1))          # SAI: đẩy trái trước -> phải ra trước
        if x.right: st.append((x.right, d + 1))
    return True

r = build_tree(examples[0][0])
print("sai :", wrong_dfs_right_first(r), "| đúng:", Solution().isEvenOddTree(r))

sai : False | đúng: True


**Vì sao sai:** với stack, phần tử đẩy sau ra trước; đẩy trái trước nghĩa là thăm **phải trước** ⇒ mỗi tầng được duyệt từ phải sang trái ⇒ chiều tăng/giảm bị đảo. **Sửa:** đẩy phải trước, trái sau (hoặc DFS đệ quy gọi trái trước).

### Lỗi 5 — DFS đệ quy trên cây sâu

In [23]:
def recursive_dfs(root):
    last = []
    def go(x, d):
        if x is None: return True
        v = x.val
        if v % 2 == d % 2: return False
        if d == len(last): last.append(v)
        elif (d % 2 == 0 and v <= last[d]) or (d % 2 == 1 and v >= last[d]): return False
        else: last[d] = v
        return go(x.left, d + 1) and go(x.right, d + 1)
    return go(root, 0)

deep = TreeNode(1); cur = deep
for d in range(1, 5000):
    cur.left = TreeNode(2 * d + (0 if d % 2 else 1)); cur = cur.left
try:
    print(recursive_dfs(deep))
except RecursionError:
    print("DFS đệ quy trên cây sâu 5000 -> RecursionError | BFS:", Solution().isEvenOddTree(deep))

DFS đệ quy trên cây sâu 5000 -> RecursionError | BFS: True


**Vì sao sai:** logic đúng nhưng Python giới hạn độ sâu đệ quy (~1000–3000); đề cho tới `10⁵` nút nên cây lệch có thể sâu `10⁵`. **Sửa:** BFS hoặc DFS bằng stack tường minh.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Điều kiện "**theo từng tầng**, từ trái sang phải" ⇒ BFS theo tầng là lựa chọn tự nhiên nhất.
- "Cả dãy tăng/giảm nghiêm ngặt" ⇒ chỉ cần so **phần tử liền trước** — giữ một biến `prev`, nhớ **reset** khi sang dãy mới.
- Ghi chép lại điều kiện bằng một bảng nhỏ (tầng chẵn → lẻ & tăng; tầng lẻ → chẵn & giảm) trước khi code để không nhầm chiều.
- Mẹo `sign = ±1` biến "tăng hoặc giảm" thành một phép so sánh duy nhất.
- Giá trị trong khoảng `[1, 10⁶]` ⇒ có thể khởi tạo `prev` bằng `0` / `10⁶ + 1` thay vì `±inf`.

## 15. Biến thể của bài

1. **LC 102 / 107 — Level Order Traversal (I / II)**: nền tảng duyệt theo tầng.
2. **LC 103 — Zigzag Level Order Traversal**: đổi chiều mỗi tầng — giống "tăng ở tầng chẵn, giảm ở tầng lẻ".
3. **LC 515 — Find Largest Value in Each Tree Row**, **LC 637 — Average of Levels**: thống kê theo tầng.
4. **Đảo luật**: tầng chẵn số chẵn giảm, tầng lẻ số lẻ tăng (bài tập 5 — hàm tổng quát có tham số).
5. **Trả về vị trí vi phạm đầu tiên** (bài tập 4) — hữu ích khi debug dữ liệu.
6. **Kiểm tra tầng "gần như" tăng** (cho phép bằng nhau) — đổi `<=` thành `<`.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Gom tầng | lấy mọi tầng rồi kiểm tra từng dãy | O(n) | O(n) | baseline, dễ kiểm chứng |
| 2. BFS + prev | kiểm tra ngay, reset prev mỗi tầng | O(n) | O(w) | **nộp bài** |
| 3. DFS + last[] | trái trước, `last[d]` là nút trước trong tầng | O(n) | O(h) | cây rộng, muốn ít bộ nhớ |
| 4. BFS (nút, tầng) + dấu | một phép so sánh nhờ `sign` | O(n) | O(w) | code gọn, ít nhánh |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `[1,10,4]`; (b) `[1,4,10]`; (c) `[7,6,2,3,5]`; (d) `[13,34,32,23,25,27,29]`.

**Bài 2 (tính tay):** Liệt kê các tầng của `[11,18,14,3,7,null,null,null,null,18,6]` và cho biết cây có phải Even-Odd không, vì sao.

**Bài 3 (điền TODO):** Hoàn thiện `level_ok(vals, d)` kiểm tra một tầng (list giá trị, chỉ số tầng `d`) có thỏa luật Even-Odd không.

**Bài 4 (tự viết lại):** Viết `first_violation(root)` trả `(tầng, vị trí trong tầng)` của nút **đầu tiên** (theo BFS) vi phạm, hoặc `None` nếu cây hợp lệ.

**Bài 5 (mini challenge):** Viết `is_level_rule_tree(root, even_parity, even_increasing)` tổng quát: ở tầng chẵn, mọi giá trị có `v % 2 == even_parity` và tăng (nếu `even_increasing`) hoặc giảm; tầng lẻ thì ngược lại cả hai. Bài gốc ứng với `(1, True)`.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def level_ok(vals, d):
    # TODO: tầng chẵn: mọi v lẻ và tăng nghiêm ngặt
    # TODO: tầng lẻ: mọi v chẵn và giảm nghiêm ngặt
    pass

In [25]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def first_violation(root):
    # TODO
    pass

def is_level_rule_tree(root, even_parity, even_increasing):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: viết các tầng. Mức 2: kiểm tra chẵn/lẻ trước, thứ tự sau. Mức 3: (a) `True`; (b) `False` (tầng 1 tăng); (c) `[7]`, `[6,2]`, `[3,5]` ⇒ `True`; (d) `[13]`, `[34,32]`, `[23,25,27,29]` ⇒ `True`.

**Bài 2.** Mức 1: tầng 0 `[11]`, tầng 1 `[18,14]`, tầng 2 `[3,7]`. Mức 2: tầng 3 `[18,6]`. Mức 3: mọi tầng đạt ⇒ `True`.

**Bài 3.** Mức 1: tách hai trường hợp `d % 2`. Mức 2: dùng `all(...)` cho chẵn/lẻ và cho cặp liền kề. Mức 3: dãy 1 phần tử luôn đạt thứ tự.

**Bài 4.** Mức 1: BFS như Cách 2, thêm biến đếm vị trí `k` trong tầng. Mức 2: khi vi phạm trả `(level, k)`. Mức 3: đối chiếu với vét cạn trên danh sách tầng.

**Bài 5.** Mức 1: tầng `d` chẵn: parity cần `even_parity`, tầng lẻ: `1 − even_parity`. Mức 2: hướng: `inc = even_increasing if d chẵn else not even_increasing`. Mức 3: dùng `sign = 1 if inc else −1` như Cách 4.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1, 2
b1 = [Solution().isEvenOddTree(build_tree(a)) for a in
      ([1, 10, 4], [1, 4, 10], [7, 6, 2, 3, 5], [13, 34, 32, 23, 25, 27, 29])]
print("Bài 1:", b1)
assert b1 == [True, False, True, True]
arr2 = [11, 18, 14, 3, 7, None, None, None, None, 18, 6]
print("Bài 2 các tầng:", levels(build_tree(arr2)))
assert trace_bfs(build_tree(arr2)) is True

Bài 1: [True, False, True, True]
Bài 2 các tầng: [[11], [18, 14], [3, 7], [18, 6]]
-- tầng 0 (chẵn: cần lẻ & tăng), prev = 0
   nút 11: chẵn/lẻ ✓, so với prev=0: ✓
-- tầng 1 (lẻ: cần chẵn & giảm), prev = inf
   nút 18: chẵn/lẻ ✓, so với prev=inf: ✓
   nút 14: chẵn/lẻ ✓, so với prev=18: ✓
-- tầng 2 (chẵn: cần lẻ & tăng), prev = 0
   nút 3: chẵn/lẻ ✓, so với prev=0: ✓
   nút 7: chẵn/lẻ ✓, so với prev=3: ✓
-- tầng 3 (lẻ: cần chẵn & giảm), prev = inf
   nút 18: chẵn/lẻ ✓, so với prev=inf: ✓
   nút 6: chẵn/lẻ ✓, so với prev=18: ✓
=> True


In [27]:
# Bài 3
def level_ok_sol(vals, d):
    if d % 2 == 0:
        return all(v % 2 == 1 for v in vals) and all(vals[i] < vals[i + 1] for i in range(len(vals) - 1))
    return all(v % 2 == 0 for v in vals) and all(vals[i] > vals[i + 1] for i in range(len(vals) - 1))

assert level_ok_sol([3, 7, 9], 2) and not level_ok_sol([3, 3, 7], 2)
assert level_ok_sol([12, 8, 6, 2], 3) and not level_ok_sol([9, 1], 1)
rnd = random.Random(3)
for _ in range(500):
    root = assign_valid_values(random_shape(rnd.randint(1, 30), rnd), rnd, 200)
    if rnd.random() < 0.5:
        root.val = rnd.randint(1, 40)
    lv = levels(root)
    assert all(level_ok_sol(v, d) for d, v in enumerate(lv)) == Solution().isEvenOddTree(root)
print("Bài 3: OK")

Bài 3: OK


In [28]:
# Bài 4
def first_violation_sol(root):
    q = deque([root]); d = 0
    while q:
        prev = 0 if d % 2 == 0 else float("inf")
        for k in range(len(q)):
            x = q.popleft(); v = x.val
            if d % 2 == 0 and (v % 2 == 0 or v <= prev):
                return (d, k)
            if d % 2 == 1 and (v % 2 == 1 or v >= prev):
                return (d, k)
            prev = v
            if x.left: q.append(x.left)
            if x.right: q.append(x.right)
        d += 1
    return None

def first_violation_brute(root):
    for d, vals in enumerate(levels(root)):
        for k in range(len(vals)):
            if not level_ok_sol(vals[:k + 1], d):
                return (d, k)
    return None

assert first_violation_sol(build_tree(examples[1][0])) == (2, 1)
assert first_violation_sol(build_tree(examples[2][0])) == (1, 0)
assert first_violation_sol(build_tree(examples[0][0])) is None
rnd = random.Random(4)
for _ in range(600):
    root = assign_valid_values(random_shape(rnd.randint(1, 30), rnd), rnd, 200)
    nodes, st = [], [root]
    while st:
        x = st.pop(); nodes.append(x); st += [c for c in (x.left, x.right) if c]
    for _ in range(rnd.randint(0, 2)):
        rnd.choice(nodes).val = rnd.randint(1, 40)
    assert first_violation_sol(root) == first_violation_brute(root)
print("Bài 4: OK")

Bài 4: OK


In [29]:
# Bài 5
def is_level_rule_tree_sol(root, even_parity, even_increasing):
    q = deque([(root, 0)]); cur, prev = -1, None
    while q:
        x, d = q.popleft(); v = x.val
        need_par = even_parity if d % 2 == 0 else 1 - even_parity
        inc = even_increasing if d % 2 == 0 else not even_increasing
        if v % 2 != need_par:
            return False
        sign = 1 if inc else -1
        if d != cur:
            cur, prev = d, v
        elif sign * v <= sign * prev:
            return False
        else:
            prev = v
        if x.left: q.append((x.left, d + 1))
        if x.right: q.append((x.right, d + 1))
    return True

def rule_brute(root, even_parity, even_increasing):
    for d, vals in enumerate(levels(root)):
        par = even_parity if d % 2 == 0 else 1 - even_parity
        inc = even_increasing if d % 2 == 0 else not even_increasing
        if any(v % 2 != par for v in vals):
            return False
        pairs = list(zip(vals, vals[1:]))
        if inc and any(a >= b for a, b in pairs): return False
        if not inc and any(a <= b for a, b in pairs): return False
    return True

rnd = random.Random(5)
for _ in range(400):
    root = random_shape(rnd.randint(1, 20), rnd)
    st = [root]
    while st:
        x = st.pop(); x.val = rnd.randint(1, 30); st += [c for c in (x.left, x.right) if c]
    for ep in (0, 1):
        for ei in (True, False):
            assert is_level_rule_tree_sol(root, ep, ei) == rule_brute(root, ep, ei)
    assert is_level_rule_tree_sol(root, 1, True) == Solution().isEvenOddTree(root)
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template BFS kiểm tra luật theo tầng:**

```python
q, even = deque([root]), True
while q:
    prev = 0 if even else float('inf')        # reset mỗi tầng
    for _ in range(len(q)):
        x = q.popleft(); v = x.val
        if even and (v % 2 == 0 or v <= prev): return False
        if not even and (v % 2 == 1 or v >= prev): return False
        prev = v
        if x.left: q.append(x.left)
        if x.right: q.append(x.right)
    even = not even
return True
```

**Checklist:**
- [ ] Tầng chẵn ⇒ giá trị **lẻ**, tăng; tầng lẻ ⇒ giá trị **chẵn**, giảm (không nhầm chiều)?
- [ ] So sánh **nghiêm ngặt** (`<=` / `>=` là vi phạm)?
- [ ] Reset `prev` ở đầu mỗi tầng?
- [ ] Đẩy con trái trước con phải (BFS); DFS stack thì đẩy phải trước?
- [ ] Không dùng đệ quy sâu (cây lệch `10⁵`)?
- [ ] Đã test: 1 nút lẻ/chẵn, giá trị bằng nhau trong tầng, sai chẵn/lẻ, sai thứ tự, cây lệch?